# GFS Trajectory Grid over the Western US

This notebook demonstrates how to release a grid of trajectories at 500 hPa over the Western United States using a specific GFS (Global Forecast System) forecast from a public S3 bucket. It uses the `cfgrib` engine and `fsspec` for efficient data access.

In [ ]:
# Install necessary packages
!pip install s3fs hvplot cartopy cfgrib

In [ ]:
import fsspec
import xarray as xr
import numpy as np
import pandas as pd
import hvplot.xarray
import cartopy.crs as ccrs

from plat.core import run_trajectory

In [ ]:
# 1. Define the S3 URL for a specific GFS file
# Using a fixed, historical date ensures this example is always runnable.
date = "20231025"
cycle = "00"
forecast_hour = "048"
file_path = f"gfs.{date}/{cycle}/atmos/gfs.t{cycle}z.pgrb2.0p25.f{forecast_hour}"
full_s3_url = f"s3://noaa-gfs-bdp-pds/{file_path}"

# 2. Open the S3 file with fsspec and then with xarray
s3_file = fsspec.open(full_s3_url, mode='rb', s3={'anon': True})
gfs_ds = xr.open_dataset(
    s3_file.open(),
    engine="cfgrib",
    backend_kwargs={'filter_by_keys': {'typeOfLevel': 'isobaricInhPa'}},
)

# 3. Inspect the data
print(gfs_ds)

In [ ]:
# Standardize the dataset for the trajectory model
# The model expects coordinates 'lat', 'lon', 'level', 'time'
# and variables 'u', 'v', 'w'.
velocity_field = gfs_ds[['u', 'v', 'w']].rename(
    {'isobaricInhPa': 'level'}
).sel(level=500).load()

# Ensure longitude is in -180 to 180 range
velocity_field['longitude'] = (velocity_field['longitude'] + 180) % 360 - 180
velocity_field = velocity_field.rename({'longitude': 'lon'}).sortby('lon')

print(velocity_field)

In [ ]:
# Define a grid of starting points over the Western US
lat_range = np.arange(32, 50, 4)
lon_range = np.arange(-125, -110, 4)
start_lon, start_lat = np.meshgrid(lon_range, lat_range)

starting_points = {
    'lat': start_lat.flatten(),
    'lon': start_lon.flatten(),
    'level': [500] * len(start_lat.flatten()),
    'time': pd.Timestamp(velocity_field.time.values)
}

In [ ]:
# Run the trajectory model for a 2-day forecast (48 steps with dt=1 hour)
trajectory_ds = run_trajectory(
    starting_points,
    velocity_field,
    num_steps=48,
    dt=1.0
)
print(trajectory_ds)

In [ ]:
# Visualize the trajectories
trajectory_ds.hvplot.line(
    x='lon',
    y='lat',
    geo=True,
    tiles='OSM',
    crs=ccrs.PlateCarree(),
    groupby='particle',
    width=800,
    height=600,
    title='48-Hour Trajectories at 500 hPa from GFS'
)